###  Applying Appropriate Transformation

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
import datetime


def transform_to_silver(bronze_df, log_metrics=True):
    """
    Transform bronze layer Spotify data into silver layer format.
    
    Parameters:
    -----------
    bronze_df : DataFrame
        The source DataFrame from the bronze layer
    log_metrics : bool
        Whether to log data quality metrics
        
    Returns:
    --------
    DataFrame
        The transformed silver layer DataFrame
    """
    

    
    # Track metrics for data quality reporting
    metrics = {}
    
    # STEP 1: Schema Enforcement
    # --------------------------
    
    # Data type conversion
    silver_df = bronze_df \
        .withColumn("is_explicit", F.when(F.col("is_explicit") == "True", True)
                                   .when(F.col("is_explicit") == "False", False)
                                   .otherwise(None)) \
        .withColumn("snapshot_date", F.to_date(F.col("snapshot_date"))) \
        .withColumn("album_release_date", F.to_date(F.col("album_release_date"))) \
        .withColumn("daily_rank", F.col("daily_rank").cast("integer")) \
        .withColumn("daily_movement", F.col("daily_movement").cast("integer")) \
        .withColumn("weekly_movement", F.col("weekly_movement").cast("integer")) \
        .withColumn("popularity", F.col("popularity").cast("integer")) \
        .withColumn("duration_ms", F.col("duration_ms").cast("integer")) \
        .withColumn("danceability", F.col("danceability").cast("double")) \
        .withColumn("energy", F.col("energy").cast("double")) \
        .withColumn("key", F.col("key").cast("integer")) \
        .withColumn("loudness", F.col("loudness").cast("double")) \
        .withColumn("mode", F.col("mode").cast("integer")) \
        .withColumn("speechiness", F.col("speechiness").cast("double")) \
        .withColumn("acousticness", F.col("acousticness").cast("double")) \
        .withColumn("instrumentalness", F.col("instrumentalness").cast("double")) \
        .withColumn("liveness", F.col("liveness").cast("double")) \
        .withColumn("valence", F.col("valence").cast("double")) \
        .withColumn("tempo", F.col("tempo").cast("double")) \
        .withColumn("time_signature", F.col("time_signature").cast("integer"))
    
    # STEP 2: Data Cleansing
# Handle missing values
    silver_df = silver_df \
        .withColumn("country", F.when(F.col("country").isNull() | (F.col("country") == ""), "Global")
                              .otherwise(F.col("country")))
    
    # Track null values for monitoring
    if log_metrics:
        for column in silver_df.columns:
            null_count = silver_df.filter(F.col(column).isNull()).count()
            metrics[f"null_count_{column}"] = null_count
    
    # Deduplication - keep only one record per spotify_id, snapshot_date, country
    window_spec = Window.partitionBy("spotify_id", "snapshot_date", "country").orderBy("daily_rank")
    
    silver_df = silver_df \
        .withColumn("row_number", F.row_number().over(window_spec)) \
        .filter(F.col("row_number") == 1) \
        .drop("row_number")
    
    # Value range validation and correction
    silver_df = silver_df \
        .withColumn("popularity", F.when(F.col("popularity") < 0, 0)
                               .when(F.col("popularity") > 100, 100)
                               .otherwise(F.col("popularity"))) \
        .withColumn("danceability", F.when(F.col("danceability") < 0, 0)
                                  .when(F.col("danceability") > 1, 1)
                                  .otherwise(F.col("danceability"))) \
        .withColumn("energy", F.when(F.col("energy") < 0, 0)
                            .when(F.col("energy") > 1, 1)
                            .otherwise(F.col("energy"))) \
        .withColumn("speechiness", F.when(F.col("speechiness") < 0, 0)
                                 .when(F.col("speechiness") > 1, 1)
                                 .otherwise(F.col("speechiness"))) \
        .withColumn("acousticness", F.when(F.col("acousticness") < 0, 0)
                                  .when(F.col("acousticness") > 1, 1)
                                  .otherwise(F.col("acousticness"))) \
        .withColumn("instrumentalness", F.when(F.col("instrumentalness") < 0, 0)
                                      .when(F.col("instrumentalness") > 1, 1)
                                      .otherwise(F.col("instrumentalness"))) \
        .withColumn("liveness", F.when(F.col("liveness") < 0, 0)
                              .when(F.col("liveness") > 1, 1)
                              .otherwise(F.col("liveness"))) \
        .withColumn("valence", F.when(F.col("valence") < 0, 0)
                             .when(F.col("valence") > 1, 1)
                             .otherwise(F.col("valence")))
    
    # STEP 3: Data Enrichment
    
    
    # Parse artists into array
    #create artist array and artist count column
    silver_df = silver_df \
        .withColumn("artists_array", F.split(F.col("artists"), ", ")) \
        .withColumn("artist_count", F.size(F.col("artists_array")))
    
    # Extract date components
    silver_df = silver_df \
        .withColumn("snapshot_year", F.year(F.col("snapshot_date"))) \
        .withColumn("snapshot_month", F.month(F.col("snapshot_date"))) \
        .withColumn("snapshot_day", F.dayofmonth(F.col("snapshot_date"))) \
        .withColumn("snapshot_weekday", F.dayofweek(F.col("snapshot_date"))) \
        .withColumn("album_release_year", F.year(F.col("album_release_date"))) \
        .withColumn("album_release_month", F.month(F.col("album_release_date"))) \
        .withColumn("album_release_day", F.dayofmonth(F.col("album_release_date"))) \
        .withColumn("track_age_days", F.datediff(F.col("snapshot_date"), F.col("album_release_date")))
    
    # Add musical key name
    silver_df = silver_df \
        .withColumn("key_name", F.when(F.col("key") == 0, "C")
                               .when(F.col("key") == 1, "C♯/D♭")
                               .when(F.col("key") == 2, "D")
                               .when(F.col("key") == 3, "D♯/E♭")
                               .when(F.col("key") == 4, "E")
                               .when(F.col("key") == 5, "F")
                               .when(F.col("key") == 6, "F♯/G♭")
                               .when(F.col("key") == 7, "G")
                               .when(F.col("key") == 8, "G♯/A♭")
                               .when(F.col("key") == 9, "A")
                               .when(F.col("key") == 10, "A♯/B♭")
                               .when(F.col("key") == 11, "B")
                               .otherwise("Unknown"))
    
    # Add mode name (Major/Minor)
    silver_df = silver_df \
        .withColumn("mode_name", F.when(F.col("mode") == 0, "Minor")
                                .when(F.col("mode") == 1, "Major")
                                .otherwise("Unknown"))
    
    # Add duration in seconds
    silver_df = silver_df \
        .withColumn("duration_sec", F.round(F.col("duration_ms") / 1000, 2))
    
    # Add audio feature categories
    silver_df = silver_df \
        .withColumn("energy_category", F.when(F.col("energy") >= 0.8, "High")
                                      .when(F.col("energy") >= 0.4, "Medium")
                                      .otherwise("Low")) \
        .withColumn("valence_category", F.when(F.col("valence") >= 0.7, "Positive")
                                       .when(F.col("valence") >= 0.3, "Neutral")
                                       .otherwise("Negative")) \
        .withColumn("tempo_category", F.when(F.col("tempo") >= 120, "Fast")
                                     .when(F.col("tempo") >= 76, "Medium")
                                     .otherwise("Slow"))
    
    # Add unique record ID
    silver_df = silver_df \
        .withColumn("silver_id", F.concat(
            F.col("spotify_id"), 
            F.lit("_"), 
            F.date_format(F.col("snapshot_date"), "yyyyMMdd"), 
            F.lit("_"),
            F.when(F.col("country") == "Global", "GLB").otherwise(F.col("country"))
        ))  

    # Drop rows that contain any null values
    silver_df = silver_df.dropna()
  


    # STEP 4: Write final stats and store in silver layer
    # -------------------------
    container = "silver-layer"
    storage_account = "etlprojectspotify"
    mount_name = "silver"
    result=None

    result=dbutils.notebook.run("/Workspace/Users/suman.kr.ghorai@gmail.com/Spotify-ETL-Databricks/utils/mount_utils", 60, {
    "container": container,
    "storage_account": storage_account,
    "mount_name": mount_name
    })
    
    if result == None:
        print("Mounting failed")
    else:
        print("Mounting successful")    
    
    
    silver_df.write.format("delta").mode("overwrite").save("/mnt/silver/spotify_silver")

    if log_metrics:
        metrics["input_record_count"] = bronze_df.count()
        metrics["output_record_count"] = silver_df.count()
        metrics["total_columns"] = len(silver_df.columns)
        metrics["total_null_values"] = sum(silver_df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in silver_df.columns]).first())

        metrics["total_distinct_records"] = silver_df.select("silver_id").distinct().count()
        

        

        metrics["duplicate_records_removed"] = metrics["input_record_count"] - metrics["output_record_count"]
        print(f"Transformation metrics: {metrics}")
    
    
    
    return silver_df, metrics  # Return transformed DataFrame and metrics












In [0]:
df_sql= spark.read.format("parquet") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/mnt/bronze/sql_data/*.parquet")

df_csv= spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/mnt/bronze/csv_data/*.csv")

display(df_csv.count())
display(df_sql.count()) 
len(df_sql.columns)

df = df_sql.unionByName(df_csv)
df.count()
try:
    silver_df, metrics = transform_to_silver(df)
except Exception as e:
    print(f"Error: {e}")
    dbutils.notebook.exit(f"Failed :{e}")



if metrics:
    dbutils.notebook.exit(f"Success:\n{metrics}")
else:
    dbutils.notebook.exit("Failed")

In [0]:
display(metrics)